In [13]:
STOCK_IDS: list[str] = [
    "1303", "4958", "6505", "8046", "3189", "4938", "1301", "3533", "1802", "6415",
    "1326", "2603", "2368", "2891", "3034", "3037", "2409", "3044", "3653", "2379",
    "2383", "2881", "3231", "2609", "2313", "6446", "2882", "2357", "3443", "3711",
    "2317", "3008", "2345", "2404", "2330", "2887", "3661", "2376", "2382", "2883",
]

# 設定全域股票範圍（whitelist 限制在這 40 檔）
# 之後所有 data.get() 與 data.indicator() 只會下載並回傳這 40 檔的資料
data.set_universe(whitelist=STOCK_IDS)

In [1]:
"""Colab cell: API-key authentication and 2330 indicator checks."""
import contextlib, io, os, json, zipfile
from getpass import getpass
from pathlib import Path
import numpy as np
import pandas as pd
import finlab
from finlab import data
from IPython.display import display

finlab.login()  # 憑證已快取時不會再開瀏覽器

已登入（使用快取憑證）。


In [3]:
START_DATE = "2026-07-01"

1. 加權指數收盤價

In [4]:
def taiex_close_latest():

    # 取得加權指數收盤資料
    p = data.get("taiex_total_index:收盤指數", start=START_DATE)

    # 最新收盤指數，單位：點
    taiex_close = (p.iloc[[-1]]).round(2)

    return taiex_close
taiex_close_latest()

Your version is 2.2.16, please install a newer version.
Use "pip install finlab==2.2.17" to update the latest version.


symbol         TAIEX
date                
2026-10-22  46041.43

2. 加權指數20日報酬

In [5]:
def taiex_return_20d():

    # 取得加權指數收盤資料
    p = data.get("taiex_total_index:收盤指數", start=START_DATE)

    # 近20日報酬，單位：%
    taiex_return_20d_pct = ((p.iloc[[-1]] / p.iloc[[-21]] - 1) * 100).round(2)

    return taiex_return_20d_pct
taiex_return_20d()

symbol      TAIEX
date             
2026-10-22  -3.52

3. 加權指數20日均線

In [6]:
def taiex_ma20_latest():

    # 取得加權指數收盤資料
    p = data.get("taiex_total_index:收盤指數", start=START_DATE)

    # 最新20日均線，單位：點
    taiex_ma20 = (p.rolling(20).mean().iloc[[-1]]).round(2)

    return taiex_ma20
taiex_ma20_latest()

symbol         TAIEX
date                
2026-10-22  48071.61

4. 加權指數60日均線

In [7]:
def taiex_ma60_latest():

    # 取得加權指數收盤資料
    p = data.get("taiex_total_index:收盤指數", start=START_DATE)

    # 最新60日均線，單位：點
    taiex_ma60 = (p.rolling(60).mean().iloc[[-1]]).round(2)

    return taiex_ma60
taiex_ma60_latest()

symbol         TAIEX
date                
2026-10-22  46207.09

5. 大盤20日最大回撤

In [8]:
def taiex_max_drawdown_20d():

    # 取得加權指數收盤資料
    p = data.get("taiex_total_index:收盤指數", start=START_DATE)

    # 最近20個交易日
    recent = p.iloc[-20:]

    # 期間內截至各日的最高指數
    running_high = recent.cummax()

    # 各日相對先前高點的回撤
    drawdown = (recent / running_high - 1) * 100

    # 最大回撤，單位：%；以負值表示跌幅
    taiex_max_drawdown_20d_pct = drawdown.min().to_frame().T.round(2)

    # 可選：若想把 index 標記為最新日期
    taiex_max_drawdown_20d_pct.index = [p.index[-1]]

    return taiex_max_drawdown_20d_pct
taiex_max_drawdown_20d()

symbol      TAIEX
2026-10-22  -7.59

6. 外資近20日淨買賣超金額

In [9]:
def market_foreign_net_buy_20d():

    # 取得上市市場外資淨買賣超金額
    foreign = data.get("institutional_investors_trading_all_market_summary:買賣超", start=START_DATE)["上市外資及陸資(不含外資自營商)"]

    # 最近20日淨買賣超金額，沿用原始金額單位
    foreign_net_buy_20d_amount = (foreign.rolling(20).sum().iloc[[-1]]).round(2)

    return foreign_net_buy_20d_amount
market_foreign_net_buy_20d()

date
2026-10-22   -8.030084e+11
Name: 上市外資及陸資(不含外資自營商), dtype: float64

7.投信近20日淨買賣超金額

In [10]:
def market_trust_net_buy_20d():

    # 取得上市市場外資淨買賣超金額
    trust = data.get("institutional_investors_trading_all_market_summary:買賣超", start=START_DATE)["上市投信"]

    # 最近20日淨買賣超金額，沿用原始金額單位
    trust_net_buy_20d_amount = (trust.rolling(20).sum().iloc[[-1]]).round(2)

    return trust_net_buy_20d_amount
market_trust_net_buy_20d()

date
2026-10-22    1.359977e+11
Name: 上市投信, dtype: float64

8.股票市場漲跌家數比(倍)  
這個不能使用data.set_universe(whitelist=STOCK_IDS)會報錯要額外處理

In [11]:
def market_advance_decline_ratio():

    # 取得集中市場股票上漲、下跌家數
    up = data.get("limit_up_down_count:上漲")["股票"]
    down = data.get("limit_up_down_count:下跌")["股票"]

    # 對齊交易日期
    down = down.reindex(up.index)

    # 最新漲跌家數
    up_count = up.iloc[[-1]]
    down_count = down.iloc[[-1]]

    # 漲跌家數比，單位：倍
    advance_decline_ratio = (up_count / down_count).round(2)

    return advance_decline_ratio
market_advance_decline_ratio()

date
2026-10-22    0.33
Name: 股票, dtype: float64

9. 上市市場融資餘額近5日變化率

In [12]:
def listed_margin_balance_change_5d():

    # 取得上市市場融資金額餘額
    margin = data.get("margin_balance:融資券總餘額", start=START_DATE)["上市融資交易金額"]

    # 近5個交易日變化率，單位：%
    margin_balance_change_5d_pct = (((margin.iloc[[-1]] / margin.iloc[-6]) - 1) * 100).round(2)

    return margin_balance_change_5d_pct
listed_margin_balance_change_5d()

date
2026-10-22   -1.94
Name: 上市融資交易金額, dtype: float64